# Loading of important libraries

In [8]:
import sys
import os
import pickle
from statistics import mean
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, randint, uniform
from rdkit import Chem
from rdkit.Chem import (Descriptors,rdMolDescriptors,PandasTools,AllChem)
from boruta import BorutaPy
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import (train_test_split,RandomizedSearchCV,GridSearchCV,KFold,cross_validate)
from sklearn.metrics import (make_scorer,classification_report,confusion_matrix,ConfusionMatrixDisplay,roc_auc_score,f1_score,roc_curve,precision_score,recall_score,accuracy_score,cohen_kappa_score,matthews_corrcoef,r2_score)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

# loading of calculated discriptors

In [9]:
import pandas as pd
import numpy as np
Location = r'C:\Users\wanih\\NLRP_3\\Dataset_training_descriptors_NLRP3.csv'
df = pd.read_csv(Location)
df.head()

,Molecule ChEMBL ID,Activity,Smiles,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CHEMBL5419955,1,Cc1cc(NC2CCCN(C3CCC3)C2)nnc1-c1ccc2sccc2c1O,10.674374,10.674374,0.291926,0.291926,0.665444,20.964286,394.544,...,0,0,0,0,0,0,0,1,0,0
1,CHEMBL5429355,1,CC(C)c1nn(CC(=O)N[C@@H]2CCCN(C3CCC3)C2)c(=O)c2...,13.134365,13.134365,0.060946,-0.251101,0.629496,20.580645,462.019,...,0,0,0,0,0,0,0,1,0,0
2,CHEMBL5409389,1,CC(C)OC(=O)[C@@H](Cc1ncccn1)OC(=O)Nc1c2c(cc3c1...,12.821105,12.821105,0.060335,-1.111473,0.733833,15.433333,409.486,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL5435222,1,Cc1cc(N[C@@H]2CCCN(C)C2)nnc1-c1ccc(C2CC2)cc1O,10.416493,10.416493,0.302236,0.302236,0.892718,21.280000,338.455,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL5414596,1,CN1CCC[C@@H](Nc2cc(C(F)(F)F)c(-c3ccc(C(F)(F)F)...,13.589469,13.589469,0.090022,-4.847025,0.721570,18.655172,420.357,...,0,0,0,0,0,0,0,0,0,0


In [10]:
df['Activity'].value_counts()

Activity
0    356
1    265
Name: count, dtype: int64

In [11]:
X=df.iloc[:, 3:,]
y = df['Activity']

In [12]:
X

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,10.674374,10.674374,0.291926,0.291926,0.665444,20.964286,394.544,368.336,394.182732,146,...,0,0,0,0,0,0,0,1,0,0
1,13.134365,13.134365,0.060946,-0.251101,0.629496,20.580645,462.019,433.795,461.165224,166,...,0,0,0,0,0,0,0,1,0,0
2,12.821105,12.821105,0.060335,-1.111473,0.733833,15.433333,409.486,382.270,409.200156,158,...,0,0,0,0,0,0,0,0,0,0
3,10.416493,10.416493,0.302236,0.302236,0.892718,21.280000,338.455,312.247,338.210661,132,...,0,0,0,0,0,0,0,0,0,0
4,13.589469,13.589469,0.090022,-4.847025,0.721570,18.655172,420.357,402.213,420.138481,158,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
616,12.062899,12.062899,0.028674,-0.028674,0.938203,12.809524,283.327,266.191,283.120843,108,...,0,0,0,0,0,0,0,0,0,0
617,12.022082,12.022082,0.084229,-0.084229,0.798541,10.523810,282.299,268.187,282.100442,106,...,0,0,0,0,0,0,0,0,0,0
618,12.144279,12.144279,0.002549,-2.897916,0.888570,10.363636,307.296,292.176,307.102000,116,...,0,0,0,0,0,0,0,0,0,0
619,11.835078,11.835078,0.060723,-0.174383,0.827567,9.947368,257.289,242.169,257.105193,98,...,0,0,0,0,0,0,0,0,0,0


In [13]:
from sklearn.model_selection import train_test_split
#Dataset Spliting
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,stratify=y, random_state=0)

# Data Spliting Scaffolds method

In [15]:
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np

def generate_scaffold(smiles):
    """Generates the Bemis-Murcko scaffold for a given SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

# 1. Group dataframe indices by their molecular scaffold using the 'Smiles' column
scaffolds = {}
for idx, smiles in df['Smiles'].items():
    scaffold = generate_scaffold(smiles)
    if scaffold not in scaffolds:
        scaffolds[scaffold] = []
    scaffolds[scaffold].append(idx)

# 2. Sort scaffolds by number of molecules (descending) to help hit the 70/30 target
scaffold_sets = sorted(scaffolds.values(), key=lambda x: (len(x), x[0]), reverse=True)

# 3. Assign to train/test using a greedy approach (targeting 70% train)
train_idx, test_idx = [], []
train_cutoff = int(0.7 * len(df)) 

for scaffold_indices in scaffold_sets:
    if len(train_idx) + len(scaffold_indices) > train_cutoff:
        test_idx.extend(scaffold_indices)
    else:
        train_idx.extend(scaffold_indices)

# 4. Map the grouped indices back to the X and y variables you created earlier
X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print(f"Scaffold Split - Train: {len(X_train)} ({len(X_train)/len(df):.1%}), Test: {len(X_test)} ({len(X_test)/len(df):.1%})")

Scaffold Split - Train: 434 (69.9%), Test: 187 (30.1%)


# Data Normalization

In [16]:
from sklearn.preprocessing import StandardScaler
scaler = RobustScaler().fit(X_train)
X_tr1 = scaler.transform(X_train)
X_xt1 = scaler.transform(X_test)
with open("standard_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [17]:
lst = X_train.columns
lst2 = X_test.columns

In [18]:
X_tr1_df = pd.DataFrame(X_tr1, columns=lst)
X_xt1_df = pd.DataFrame(X_xt1, columns=lst2)

# Features Selection using boruta 

In [19]:
# Step 1: Initialize Random Forest Classifier
rf = RandomForestClassifier(n_jobs=-1, random_state=0)
# Step 2: Initialize Boruta
boruta_selector = BorutaPy(rf, n_estimators='auto', random_state=0)
# Step 3: Fit Boruta on the data
boruta_selector.fit(X_tr1_df.values, y_train.values)
# Step 4: Get selected features
selected_features = X_tr1_df.columns[boruta_selector.support_]
# Convert the selected features to a DataFrame to save as Excel
selected_features_df = pd.DataFrame(selected_features, columns=['Selected Features'])
# Save the selected features to an Excel file
selected_features_df.to_excel('selected_features.xlsx', index=False)
selected_features_df
#features to be used for the model devolpment
X_train_selected = X_tr1_df[selected_features]
X_test_selected = X_xt1_df[selected_features]
# Step 5: Train Random Forest on selected features
rf.fit(X_train_selected[selected_features], y_train)
feature_importances = rf.feature_importances_
# Create a DataFrame of selected features and their importances
importance_df = pd.DataFrame({
    'Selected Features': selected_features,
    'Feature Importance': feature_importances
}).sort_values(by='Feature Importance', ascending=False)
# Round feature importances to two decimal places
importance_df['Feature Importance'] = importance_df['Feature Importance'].round(3)
# Display the DataFrame
#save the dataframe
importance_df.to_csv('importance_df.csv',index=False)

In [20]:
# Convert Index to list (important for portability)
selected_features_list = list(selected_features)

with open("selected_features.pkl", "wb") as f:
    pickle.dump(selected_features_list, f)

# Hyperparameter tunning

# 1.DECISION TREE

In [21]:
model1 = DecisionTreeClassifier(class_weight='balanced',random_state=0)
max_depth = list(range(1,30))
max_features=list(range(1,30))
tuning_parameters = dict(max_depth=max_depth,max_features=max_features)
grid_search = GridSearchCV(model1,tuning_parameters,scoring='roc_auc', cv=5, n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_selected,y_train)

# Print the best parameters found by the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: ",grid_search.best_score_)

Best parameters: {'max_depth': 2, 'max_features': 5}
Best cross-validation score:  0.8313004812722328


# 2.KNeighborsClassifier

In [22]:
model2 = KNeighborsClassifier(weights='uniform')
leaf_size =list(range(1,10))
n_neighbors =list(range(1,50))
metric = ['minkowski', 'euclidean', 'manhattan']
tuning_parameters = dict(leaf_size=leaf_size, n_neighbors=n_neighbors,metric= metric)
#grid search
grid_search=GridSearchCV(model2,tuning_parameters,cv=5,scoring='roc_auc', n_jobs=-1,)

# Fit the grid search to the training data
grid_search.fit(X_train_selected,y_train)

# Print the best parameters found by the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: ",grid_search.best_score_)

Best parameters: {'leaf_size': 1, 'metric': 'manhattan', 'n_neighbors': 11}
Best cross-validation score:  0.8295553463067588


# 3.Support Vector Machines (SVM)

In [23]:
model3 = SVC(random_state=0)
#tunning
kernel=['rbf','linear']
C=[0.1,0.01,0.001,1]
gamma=[0.1,0.01,0.001,1]
tuning_parameters = dict( C=C,gamma=gamma,kernel=kernel)

grid_search = GridSearchCV(model3,tuning_parameters,cv=5,scoring='roc_auc', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_selected,y_train)

# Print the best parameters found by the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: ",grid_search.best_score_)

Best parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'linear'}
Best cross-validation score:  0.8448127223268467


# 4.Random Forest

In [24]:
model4= RandomForestClassifier(max_depth=5,random_state=0)
#tunning
# Define hyperparameters to tune
n_estimators = list(range(1,50))
criterion = ['gini', 'entropy']
max_features = list(range(1,20))

# Create the parameter grid for GridSearchCV
tuning_parameters = dict(n_estimators=n_estimators, criterion=criterion,max_features=max_features)
grid_search = GridSearchCV(model4,tuning_parameters,cv=5,scoring='roc_auc', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_selected,y_train)
# Print the best parameters found by the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: ",grid_search.best_score_)

Best parameters: {'criterion': 'entropy', 'max_features': 18, 'n_estimators': 23}
Best cross-validation score:  0.8642331031596567


# 5. AdaBoost decision tree (ABDT)

In [25]:
# Initialize a base learner (DecisionTreeClassifier with max depth 1)
base_learner = DecisionTreeClassifier(random_state=0,max_depth=3)

# Initialize AdaBoost with the base estimator (use 'estimator' instead of 'base_estimator')
model6 = AdaBoostClassifier(estimator=base_learner,algorithm='SAMME', random_state=0)

# Define the hyperparameter grid for grid search
n_estimators = [50, 100, 200]  # Number of weak learners
learning_rate = [0.01, 0.1, 1]  # Learning rate

# Tuning parameters
tuning_parameters = dict(n_estimators=n_estimators, 
                         learning_rate=learning_rate)

# Initialize GridSearchCV
grid_search = GridSearchCV(model6, tuning_parameters, cv=5, scoring='roc_auc', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_selected, y_train)

# Print the best parameters found by the grid search
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

C:\Users\wanih\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


Best parameters: {'learning_rate': 0.1, 'n_estimators': 200}
Best cross-validation score:  0.8460504289600334


# Models and Performance matricss and Performance matrics

In [26]:
# Define models
models = []
models.append(('DT', DecisionTreeClassifier(class_weight='balanced',random_state=0,max_depth=6, max_features=14)))
models.append(('KNN',KNeighborsClassifier(weights='uniform',leaf_size=1, n_neighbors=10,metric='manhattan')))
models.append(('SVM',SVC(random_state=0,C=1,gamma=0.1,kernel='rbf')))
models.append(('RF', RandomForestClassifier(max_depth=5,random_state=0,criterion='entropy',n_estimators=16,max_features=21)))
base_learner = DecisionTreeClassifier(random_state=0,max_depth=3)
models.append(('ADBT',AdaBoostClassifier(estimator=base_learner,algorithm='SAMME', random_state=0,learning_rate= 0.1,n_estimators=200)))

# Initialize result lists and names list
result1_train = []  # For storing training ACCURACY results
result2_train = []  # For storing training ROC_AUC results
result3_train = []  # For storing training PRECISION results
result4_train = []  # For storing training RECALL results
result5_train = []  # For storing training SPECIFICITY results
result6_train = []  # For storing training F1_SCORE results
result7_train = []  # For storing training MCC results
result1_test = []   # For storing test ACCURACY results
result2_test = []   # For storing test ROC_AUC results
result3_test = []   # For storing test PRECISION results
result4_test = []   # For storing test RECALL results
result5_test = []   # For storing test SPECIFICITY results
result6_test = []   # For storing test F1_SCORE results
result7_test = []   # For storing test MCC results

names = []  # For storing model names

# Loop through models and calculate metrics for both training and test sets
for name, model in models:
    # Fit the model on the training data
    model.fit(X_train_selected, y_train)
    
    # Get predictions for the training and test sets
    y_train_pred = model.predict(X_train_selected)
    y_test_pred = model.predict(X_test_selected)
    
    # Calculate metrics for the training set
    accuracy_train = accuracy_score(y_train, y_train_pred)
    auc_roc_train = roc_auc_score(y_train, y_train_pred)
    f1_train = f1_score(y_train, y_train_pred)  # F1-score for training set
    recall_train = recall_score(y_train, y_train_pred)
    precision_train = precision_score(y_train, y_train_pred)
    mcc_train = matthews_corrcoef(y_train, y_train_pred)  # MCC for training set
    # Calculate specificity for the training set
    tn_train, fp_train, fn_train, tp_train = confusion_matrix(y_train, y_train_pred).ravel()
    specificity_train = tn_train / (tn_train + fp_train)
    
    # Append the results to the training result lists
    result1_train.append(round(accuracy_train * 100, 2))         # Accuracy as percentage
    result2_train.append(round(auc_roc_train * 100, 2))           # AUC-ROC as percentage
    result3_train.append(round(f1_train, 2))                # F1-Score as percentage
    result4_train.append(round(recall_train * 100, 2))            # Recall as percentage
    result5_train.append(round(precision_train * 100, 2))         # Precision as percentage
    result6_train.append(round(mcc_train, 2))               # MCC as percentage
    result7_train.append(round(specificity_train * 100, 2))       # Specificity as percentage

    # Calculate metrics for the test set
    accuracy_test = accuracy_score(y_test, y_test_pred)
    auc_roc_test = roc_auc_score(y_test, y_test_pred)
    f1_test = f1_score(y_test, y_test_pred)  # F1-score for test set
    recall_test = recall_score(y_test, y_test_pred)
    precision_test = precision_score(y_test, y_test_pred)
    mcc_test = matthews_corrcoef(y_test, y_test_pred)  # MCC for test set

    # Calculate specificity for the test set (True Negative Rate)
    tn_test, fp_test, fn_test, tp_test = confusion_matrix(y_test, y_test_pred).ravel()
    specificity_test = tn_test / (tn_test + fp_test)
    
    # Append the results to the test result lists
    result1_test.append(round(accuracy_test * 100, 2))         # Accuracy as percentage
    result2_test.append(round(auc_roc_test * 100, 2))           # AUC-ROC as percentage
    result3_test.append(round(f1_test, 2))                # F1-Score as percentage
    result4_test.append(round(recall_test * 100, 2))            # Recall as percentage
    result5_test.append(round(precision_test * 100, 2))         # Precision as percentage
    result6_test.append(round(mcc_test, 2))               # MCC as percentage
    result7_test.append(round(specificity_test * 100, 2))       # Specificity as percentage
    
    # Append model name to the names list
    names.append(name)

# Create DataFrames for each metric for both training and test sets
r1_train = pd.DataFrame(result1_train, columns=['train_Accuracy'], index=names)
r2_train = pd.DataFrame(result2_train, columns=['train_AUC-ROC'], index=names)
r3_train = pd.DataFrame(result3_train, columns=['train_F1-Score'], index=names)
r4_train = pd.DataFrame(result4_train, columns=['train_Recall'], index=names)
r5_train = pd.DataFrame(result5_train, columns=['train_Precision'], index=names)
r6_train = pd.DataFrame(result6_train, columns=['train_MCC'], index=names)
r7_train = pd.DataFrame(result7_train, columns=['train_Specificity'], index=names)

r1_test = pd.DataFrame(result1_test, columns=['test_Accuracy'], index=names)
r2_test = pd.DataFrame(result2_test, columns=['test_AUC-ROC'], index=names)
r3_test = pd.DataFrame(result3_test, columns=['test_F1-Score'], index=names)
r4_test = pd.DataFrame(result4_test, columns=['test_Recall'], index=names)
r5_test = pd.DataFrame(result5_test, columns=['test_Precision'], index=names)
r6_test = pd.DataFrame(result6_test, columns=['test_MCC'], index=names)
r7_test = pd.DataFrame(result7_test, columns=['test_Specificity'], index=names)
# Concatenate the DataFrames for both training and test sets along axis 1
training_results= pd.concat([r1_train,r2_train,r3_train,r4_train,r5_train,r6_train,r7_train], axis=1)
test_results = pd.concat([r1_test, r2_test,r3_test,r4_test,r5_test, r6_test, r7_test], axis=1)
# Display the final DataFrame

C:\Users\wanih\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


In [27]:
training_results

,train_Accuracy,train_AUC-ROC,train_F1-Score,train_Recall,train_Precision,train_MCC,train_Specificity
DT,91.47,93.61,0.88,99.26,78.82,0.83,87.96
KNN,85.94,83.50,0.77,77.04,77.61,0.67,89.97
SVM,89.40,88.45,0.83,85.93,81.12,0.76,90.97
RF,92.40,93.26,0.89,95.56,82.69,0.83,90.97
ADBT,95.16,93.64,0.92,89.63,94.53,0.89,97.66


In [28]:
test_results

,test_Accuracy,test_AUC-ROC,test_F1-Score,test_Recall,test_Precision,test_MCC,test_Specificity
DT,67.38,68.66,0.74,65.38,84.16,0.34,71.93
KNN,70.05,76.49,0.74,60.00,95.12,0.49,92.98
SVM,71.66,76.66,0.76,63.85,93.26,0.49,89.47
RF,70.59,73.92,0.76,65.38,89.47,0.44,82.46
ADBT,65.24,71.06,0.69,56.15,90.12,0.39,85.96


# training of all model

In [19]:
DT=DecisionTreeClassifier(class_weight='balanced',random_state=0,max_depth=6, max_features=14)
KNN=KNeighborsClassifier(weights='uniform',leaf_size=1, n_neighbors=10,metric='manhattan')
SVM=SVC(random_state=0,C=1,gamma=0.1,kernel='rbf')
RF=RandomForestClassifier(max_depth=5,random_state=0,criterion='entropy',n_estimators=16,max_features=21)
ADBT=AdaBoostClassifier(estimator=base_learner,algorithm='SAMME', random_state=0,learning_rate= 0.1,n_estimators=200)

mod_dt=DT.fit(X_train_selected,y_train)
mod_knn=KNN.fit(X_train_selected,y_train)
mod_svm=SVM.fit(X_train_selected,y_train)
mod_rf=RF.fit(X_train_selected,y_train)
mod_adbt=ADBT.fit(X_train_selected,y_train)

C:\Users\wanih\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


In [20]:
import pickle

models = {
    "dt": mod_dt,
    "knn": mod_knn,
    "svm": mod_svm,
    "rf": mod_rf,
    "adbt": mod_adbt
}

for name, model in models.items():
    with open(f"{name}_model.pkl", "wb") as f:
        pickle.dump(model, f)

print("All models pickled successfully.")

All models pickled successfully.
